# ED tokenizers

Declare the alphabet, import the tokenizers, and train them, as `train_tokenizers.ipynb` does for OnHW. Every vocabulary is learned from **IAM training data only**: linguistic tokenizers from the transcriptions, handwriting tokenizers from the connectivity evidence exported by DTLR. ED only supplies its declared alphabet. Outputs are shared across WD/WI folds. Run from the TVA root with the WSL tva kernel.

Construction is frozen (`docs/DECISIONS.md`, D024): every tokenizer is trained by **TVA's own `.train()`**. A handwriting tokenizer adds one input, the letter pairs allowed to join inside a token (the most connected IAM letter pairs); that list is the only place connectivity enters. Start Jupyter with `PYTHONHASHSEED=42` (TVA breaks bigram frequency ties in `set()` order).


In [ ]:
from pathlib import Path
from tva.tokenizers import CharacterTokenizer, get_tokenizer
from tva.handwriting_tokenizers import HandwritingBigramTokenizer
from tva.handwriting_tokenizers import HandwritingBPETokenizer, HandwritingUnigramTokenizer
from tva.compression_matched_tokenizers import build_fold_tokenizers
from tva.drawn_bigram_tokenizers import build_fold_tokenizers as build_drawn_bigrams
from tva.dataset.iam import prepare_iam
from tva.handwriting.common import save_frozen

# Run from the TVA repository root with the WSL tva kernel.
iam_labels = Path('/home/artellisys/DTLR/data/IAM_new/labels.pkl')
iam_selection = Path('/home/artellisys/dtlr-output/iam-train-full/selection.json')
iam_evidence = Path('artifacts/evidence/iam/iam_connectivity_v1.json.gz')
iam_dir = 'data/tva/iam'
output = Path('artifacts/tokenizers/ed')

categories = [
    '', ' ', '!', '"', '%', "'", '(', ')', '+', ',', '-', '.', '/',
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '?',
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
    'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
    'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
    'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z',
]


def publish(trained, target):
    """Freeze a trained 0.json (and a bigram's 0.scores.json) under artifacts/tokenizers/ed."""
    for path in sorted(Path(iam_dir, 'tokenizers', trained).glob('0.*')):
        save_frozen(output / target / path.name, path.read_bytes())


## IAM in TVA format

Write the 5,694 IAM training transcriptions as a one-fold TVA dataset (`train.json`). The bigram tokenizers below train on this directory exactly like TVA trains on an OnHW dataset.

In [ ]:
prepare_iam(iam_labels, iam_selection, iam_dir)

## Character tokenizer

The character baseline uses the unchanged original TVA class and loads its standard 79-class JSON file. The file was generated with TVA’s own CharacterTokenizer.train() from the declared alphabet; no vocabulary selection from labels is involved. All five fold files contain the same vocabulary.


In [ ]:
tokenizer_char = CharacterTokenizer()
tokenizer_char.load(output / 'character' / 'char' / '0.json')
print('Character classes:', tokenizer_char.size)


## Linguistic bigram — original TVA

The unchanged TVA `BigramTokenizer`: the most frequent adjacent pairs in the unique IAM labels, greedy encoding. It also saves each bigram's text count as `0.scores.json`, which only the `dp` segmentation policy reads (D025). TVA breaks frequency ties in `set()` order, so start Jupyter with `PYTHONHASHSEED=42` to reproduce the frozen file. `save_frozen` refuses to overwrite a different existing file.

In [ ]:
tokenizer_bigram = get_tokenizer('bigram')
tokenizer_bigram.train(iam_dir, categories, 224)
publish('bigram224', 'linguistic/bigram')
print('Linguistic classes:', tokenizer_bigram.size)


## Linguistic bigram — letters only

The same TVA trainer with `letters_only=True`: pairs containing spaces, digits or punctuation are not counted, so all 145 bigrams are letter pairs, like the handwriting bigram; this is its matched partner. Spaces and punctuation stay in the categories as single-character tokens. Only frequency versus connectivity then separates the two vocabularies.

In [ ]:
tokenizer_bigram_letters = get_tokenizer('bigram')
tokenizer_bigram_letters.train(iam_dir, categories, 224, letters_only=True)
publish('bigram224_letters', 'linguistic/bigram_letters')
print('Letters-only linguistic classes:', tokenizer_bigram_letters.size)


## Linguistic BPE — original TVA, letters only

TVA's `BPETokenizer` with `letters_only=True`: merges only join letters, and spaces, digits and punctuation stay single tokens (TVA's default whitespace splitting would drop the spaces of ED sentences). 145 merges on top of the blank and the 78 characters.

In [ ]:
tokenizer_linguistic_bpe = get_tokenizer('bpe')
tokenizer_linguistic_bpe.train(iam_dir, categories, 224, letters_only=True)
publish('bpe224_letters', 'linguistic/bpe')
print('Linguistic BPE classes:', tokenizer_linguistic_bpe.size)


## Linguistic Unigram — original TVA, letters only

TVA's `UnigramTokenizer` with `letters_only=True`. Hugging Face's Unigram trainer is not bit-reproducible: retraining selects the same 145 pieces but in a different order with slightly different scores, so `save_frozen` refuses to overwrite the frozen file. Delete it deliberately to refit.

In [ ]:
tokenizer_linguistic_unigram = get_tokenizer('unigram')
tokenizer_linguistic_unigram.train(iam_dir, categories, 224, letters_only=True)
publish('unigram224_letters', 'linguistic/unigram')
print('Linguistic Unigram classes:', tokenizer_linguistic_unigram.size)


## Handwriting bigram — the most connected IAM letter pairs

TVA's `BigramTokenizer.train(..., letters_only=True)` with one extra input: only the 145 most connected letter pairs are allowed (ranked by how often the two letters are written connected, among pairs with at least 20 exactly aligned IAM observations). A bigram vocabulary of size k therefore holds the k most connected pairs, numbered by TVA in text-frequency order. `0.scores.json` stores their connected rates for the `dp` policy. No ED labels are read.


In [ ]:
tokenizer_handwriting = HandwritingBigramTokenizer()
tokenizer_handwriting.train(iam_dir, categories, 224, path_evidence=iam_evidence)
publish('handwriting_bigram224', 'handwriting/bigram_v3')
print('Handwriting classes:', tokenizer_handwriting.size)


## Handwriting BPE — TVA's BPE across connected pairs only

TVA's `BPETokenizer.train(..., letters_only=True)` with one extra input: the 145 most connected letter pairs (those written connected at least half of the time). Each IAM line is cut between two letters whose pair is not allowed before TVA's trainer sees it, so merges only form across connected pairs. Encoding is TVA's. No ED labels are read.


In [ ]:
tokenizer_bpe = HandwritingBPETokenizer()
tokenizer_bpe.train(iam_dir, categories, 224, path_evidence=iam_evidence)
publish('handwriting_bpe224', 'handwriting/bpe_v3')
print('Handwriting BPE classes:', tokenizer_bpe.size)


## Handwriting Unigram — TVA's Unigram across connected pairs only

The same as the handwriting BPE, with TVA's `UnigramTokenizer`. Like TVA's Unigram, Hugging Face's trainer is not bit-reproducible, so `save_frozen` refuses to overwrite the frozen file; delete it deliberately to refit. No ED labels are read.


In [ ]:
tokenizer_unigram = HandwritingUnigramTokenizer()
tokenizer_unigram.train(iam_dir, categories, 224, path_evidence=iam_evidence)
publish('handwriting_unigram224', 'handwriting/unigram_v3')
print('Handwriting Unigram classes:', tokenizer_unigram.size)


## Compression-matched text-only controls

TVA's letters-only tokenizer of each family trained with a smaller vocabulary, 79 + k classes, where k is chosen so that IAM text is compressed as much as by the handwriting partner. For bigram and BPE this keeps the top k bigrams or the first k merges of the full text-only tokenizer; Unigram is retrained for every k. `selection.json` records k and the whole compression curve.

The size- and compression-matched bigrams draw 145 bigrams by text frequency from the handwriting bigram's candidates and are built like the handwriting bigram, with the drawn bigrams as the allowed pairs.


In [ ]:
# Trains every k = 1..145 per family (a few minutes, mostly Unigram).
for kind in ('bigram', 'bpe', 'unigram'):
    print('Saved', build_fold_tokenizers(kind))

# Size- and compression-matched bigrams: 145 bigrams drawn by text frequency (seconds).
print('Saved', build_drawn_bigrams())


## Encode and decode


In [ ]:
text = 'In short, there will be on TV!'
for tokenizer in (tokenizer_char, tokenizer_bigram, tokenizer_bigram_letters, tokenizer_linguistic_bpe, tokenizer_linguistic_unigram, tokenizer_handwriting, tokenizer_bpe, tokenizer_unigram):
    ids = tokenizer.encode(text)
    print(type(tokenizer).__name__, ids, tokenizer.decode(ids))

Every learned tokenizer has **224 classes** (blank, 78 characters, 145 learned pieces); the compression-matched controls are smaller. Any tokenizer file loads with the config keys `bigram` (or `bigram_dp`), `bpe` and `unigram`. Their files already exist, so recognition training can use them immediately. To use an existing tokenizer without fitting, call `.load(path)`.

Detailed checks live outside this notebook:
~~~bash
python -m tva.handwriting.audit --compare --dataset-directory /mnt/c/Users/Ali/Downloads/fau-english-dataset
~~~

Recognition-model training remains separate and manual, using configs/thesis/ed/ (one folder per experiment, see its README).

## Save fold files

Publish `0.json` through `4.json` in each tokenizer directory, matching the original TVA layout. These are identical copies of the tokenizer fitted above from IAM; no ED fold labels are used. Recognition configs point to the directory and load the requested fold.


In [ ]:
from tva.handwriting.common import save_fold_tokenizers

for family, methods in {'character': ['char'], 'linguistic': ['bigram', 'bigram_letters', 'bpe', 'unigram'],
                        'handwriting': ['bigram_v3', 'bpe_v3', 'unigram_v3']}.items():
    for method in methods:
        save_fold_tokenizers(output / family / method)
